# Optimizing With UPDATE

- Generally speaking, JOINs between very large tables are very expensive regarding performance
- Steps to take for query optimization
    - Define a filtered dataset as early as possible to JOIN on smaller core population tables
    - Avoid several JOINs in a single SELECT query when large tables are involved
    - Instead, use UPDATE statements to populate fields in a temp table, one source table at a time
    - Apply indexes to fields that will be used in JOINs (will review later)

In [3]:
USE AW2019;

-- Starter Code
SELECT TOP 10
    H.SalesOrderID,
    H.OrderDate,
    D.ProductID,
    D.LineTotal,
    P.Name AS ProductName,
    SC.Name AS ProductSubcategory,
    C.Name AS ProductCategory
FROM Sales.SalesOrderHeader AS H
    JOIN Sales.SalesOrderDetail AS D
        ON H.SalesOrderID = D.SalesOrderDetailID
    JOIN Production.Product AS P
        ON D.ProductID = P.ProductID
    JOIN Production.ProductSubcategory AS SC
        ON P.ProductSubcategoryID = SC.ProductSubcategoryID
    JOIN Production.ProductCategory AS C
        ON SC.ProductCategoryID = C.ProductCategoryID
WHERE YEAR (H.OrderDate) = 2012

/*
This particular query is ok since the source tables are not large, 
but if we were working with very large tables we would be better off with the following approach.
*/

(10 rows affected)

Total execution time: 00:00:00.058

SalesOrderID,OrderDate,ProductID,LineTotal,ProductName,ProductSubcategory,ProductCategory
45266,2012-01-01 00:00:00.000,957,2384.070000,"Touring-1000 Yellow, 60",Touring Bikes,Bikes
45267,2012-01-01 00:00:00.000,707,34.990000,"Sport-100 Helmet, Red",Helmets,Accessories
45268,2012-01-01 00:00:00.000,796,2443.350000,"Road-250 Black, 58",Road Bikes,Bikes
45269,2012-01-01 00:00:00.000,933,32.600000,HL Road Tire,Tires and Tubes,Accessories
45270,2012-01-01 00:00:00.000,922,3.990000,Road Tire Tube,Tires and Tubes,Accessories
45271,2012-01-01 00:00:00.000,707,34.990000,"Sport-100 Helmet, Red",Helmets,Accessories
45272,2012-01-01 00:00:00.000,859,24.490000,"Half-Finger Gloves, M",Gloves,Clothing
45273,2012-01-01 00:00:00.000,799,1120.490000,"Road-550-W Yellow, 42",Road Bikes,Bikes
45274,2012-01-01 00:00:00.000,932,24.990000,ML Road Tire,Tires and Tubes,Accessories
45275,2012-01-01 00:00:00.000,922,3.990000,Road Tire Tube,Tires and Tubes,Accessories


In [10]:
USE AW2019;

-- Optimized Code

-- Create Filtered Temp Table for 2012 Sales Data
DROP TABLE IF EXISTS #Sales2012
CREATE TABLE #Sales2012 (
    SalesOrderID    INT,
    OrderDate       DATE
)

    INSERT INTO #Sales2012 (
        SalesOrderID,
        OrderDate
    )

    SELECT SalesOrderID, OrderDate
    FROM Sales.SalesOrderHeader
    WHERE YEAR (OrderDate) = 2012;

-- Create Master Temp Table and Insert #Sales2012
DROP TABLE IF EXISTS #ProductsSold2012
CREATE TABLE #ProductsSold2012 (
    SalesOrderID            INT,
    OrderDate               DATE,
    LineTotal               MONEY,
    ProductID               INT,
    ProductName             VARCHAR(64),
    ProductSubcategoryID    INT,
    ProductSubcategory      VARCHAR(64),
    ProductCategoryID       INT,
    ProductCategory         VARCHAR(64)
)

    INSERT INTO #ProductsSold2012 (
        SalesOrderID,
        OrderDate,
        LineTotal,
        ProductID
    )

    SELECT
        S.SalesOrderID,
        S.OrderDate,
        D.LineTotal,
        D.ProductID
    FROM #Sales2012 AS S -- less expensive join
        JOIN Sales.SalesOrderDetail AS D
            ON S.SalesOrderID = D.SalesOrderID;

-- Update Remaining NULL Values in #ProductsSold2012
UPDATE #ProductsSold2012
SET 
    ProductName = P.Name,
    ProductSubcategoryID = P.ProductSubcategoryID
FROM #ProductsSold2012 AS PS
    JOIN Production.Product AS P
        ON PS.ProductID = P.ProductID;

UPDATE #ProductsSold2012
SET 
    ProductSubcategory = SC.Name,
    ProductCategoryID = SC.ProductCategoryID
FROM #ProductsSold2012 AS PS
    JOIN Production.ProductSubcategory AS SC
        ON PS.ProductSubcategoryID = SC.ProductSubcategoryID;

UPDATE #ProductsSold2012
SET ProductCategory = C.Name
FROM #ProductsSold2012 AS PS
    JOIN Production.ProductCategory AS C
        ON PS.ProductCategoryID = C.ProductCategoryID;

-- Query Data
SELECT TOP 10 * 
FROM #ProductsSold2012;

(3915 rows affected)

(21689 rows affected)

(21689 rows affected)

(21689 rows affected)

(21689 rows affected)

(10 rows affected)

Total execution time: 00:00:01.496

SalesOrderID,OrderDate,LineTotal,ProductID,ProductName,ProductSubcategoryID,ProductSubcategory,ProductCategoryID,ProductCategory
47362,2012-07-31,259.1222,863,"Full-Finger Gloves, L",20,Gloves,3,Clothing
47362,2012-07-31,22.794,861,"Full-Finger Gloves, S",20,Gloves,3,Clothing
47362,2012-07-31,4971.4072,780,"Mountain-200 Silver, 42",1,Mountain Bikes,1,Bikes
47362,2012-07-31,109.341,815,LL Mountain Front Wheel,17,Wheels,2,Components
47362,2012-07-31,209.256,832,"ML Mountain Frame - Black, 48",12,Mountain Frames,2,Components
47362,2012-07-31,1229.4589,782,"Mountain-200 Black, 38",1,Mountain Bikes,1,Bikes
47362,2012-07-31,196.329,825,HL Mountain Rear Wheel,17,Wheels,2,Components
47362,2012-07-31,157.941,823,LL Mountain Rear Wheel,17,Wheels,2,Components
47363,2012-07-31,214.236,828,HL Road Rear Wheel,17,Wheels,2,Components
47364,2012-07-31,183.9382,738,"LL Road Frame - Black, 52",14,Road Frames,2,Components


# Improved EXISTS and NOT EXISTS w/ UPDATE
- EXISTS allows you to check for matching records from the many side of a relationship, without resulting in duplicated data from the one side
    - This works fine uinless you need additional information about the match
    - If you need to see data points pertaining to the match, UPDATE is a superior alternative
- Choosing techniques;
    - If you need to see all matches from the many side of the relationship, use JOIN
    - If you don't want to see all matches from the many side, AND don't want to see information about matches, use EXISTS
    - If you don't want to see all matches from the many side, but would like to see information about returned matches, use UPDATE


In [2]:
USE AW2019;

--Select All Orders w/ At Least One Item Over 10K, Using EXISTS
SELECT TOP 10
    H.SalesOrderID,
    H.OrderDate,
    H.TotalDue
FROM Sales.SalesOrderHeader H
WHERE EXISTS (
	SELECT H.SalesOrderID
	FROM Sales.SalesOrderDetail D
	WHERE H.SalesOrderID = D.SalesOrderID
		AND D.LineTotal > 10000
)
ORDER BY H.SalesOrderID


-- Create a Table w/ Sales Data, Including a Field for Line Total
DROP TABLE IF EXISTS #Sales
CREATE TABLE #Sales (
    SalesOrderID    INT,
    OrderDate       DATE,
    TotalDue        MONEY,
    LineTotal       MONEY
)


--Insert Sales Data into Temp Table
    INSERT INTO #Sales (
        SalesOrderID,
        OrderDate,
        TotalDue
    )

    SELECT
        SalesOrderID,
        OrderDate,
        TotalDue
    FROM Sales.SalesOrderHeader


--Update Temp Table w/ > 10K Line Totals
UPDATE #Sales
SET LineTotal = D.LineTotal
FROM #Sales A
	JOIN Sales.SalesOrderDetail D
		ON A.SalesOrderID = D.SalesOrderID
WHERE D.LineTotal > 10000


--Recreate EXISTS, Returning Records With Existing Line Total
SELECT TOP 10 * 
FROM #Sales 
WHERE LineTotal IS NOT NULL


--Recreate NOT EXISTS, Returning Records Without Existing Line Total
SELECT TOP 10 * 
FROM #Sales 
WHERE LineTotal IS NULL

(10 rows affected)

(31465 rows affected)

(416 rows affected)

(10 rows affected)

(10 rows affected)

Total execution time: 00:00:00.395

SalesOrderID,OrderDate,TotalDue
43683,2011-05-31 00:00:00.000,48204.0662
43695,2011-05-31 00:00:00.000,44344.8265
43843,2011-07-01 00:00:00.000,37106.2915
43864,2011-07-01 00:00:00.000,43335.7219
43869,2011-07-01 00:00:00.000,55408.1581
43875,2011-07-01 00:00:00.000,137343.2877
43881,2011-07-01 00:00:00.000,43706.8175
43884,2011-07-01 00:00:00.000,130416.4829
43890,2011-07-01 00:00:00.000,84686.9878
43894,2011-07-01 00:00:00.000,36585.904


SalesOrderID,OrderDate,TotalDue,LineTotal
55234,2013-08-30,54244.2389,11135.952
55239,2013-08-30,44793.3885,11135.952
55241,2013-08-30,68175.7662,11015.952
55245,2013-08-30,76499.1179,10635.2699
55249,2013-08-30,57788.8395,13769.94
55254,2013-08-30,117506.1173,19930.8252
55269,2013-08-30,55864.4473,11602.1126
55275,2013-08-30,72873.2643,12527.946
55276,2013-08-30,83900.0234,10156.146
55277,2013-08-30,64826.3865,11135.952


SalesOrderID,OrderDate,TotalDue,LineTotal
53779,2013-08-03,24.2327,NULL
53780,2013-08-03,16.5529,NULL
53781,2013-08-03,46.3327,NULL
53782,2013-08-03,2708.6865,NULL
53783,2013-08-03,2799.9927,NULL
53784,2013-08-03,2699.9018,NULL
53785,2013-08-03,861.3254,NULL
53786,2013-08-03,2.5305,NULL
53787,2013-08-03,2.5305,NULL
53788,2013-08-03,93.8808,NULL
